# Hints — Window Functions Advanced

**Level:** Medium

> These hints are here when you're stuck. Try solving each problem on your own first.
> Hints are provided for selected problems - the trickier ones. If a problem is not listed here, it is meant to be solvable from the docs and earlier problems.
> Reveal one hint at a time.

## Problem 1

<details>
<summary>Hint 1</summary>

Define a single window spec with `Window.partitionBy("franchiseID").orderBy(F.col("totalPrice").desc())`, then apply both `F.rank()` and `F.dense_rank()` over it.

</details>

<details>
<summary>Hint 2</summary>

Use `.withColumn("rnk", F.rank().over(w)).withColumn("dense_rnk", F.dense_rank().over(w))` and then `.select()` to keep only the 5 expected columns.

</details>

## Problem 2

<details>
<summary>Hint 1</summary>

`F.ntile(4)` splits the partition into 4 roughly equal buckets. Apply it over a window partitioned by `franchiseID` and ordered by `totalPrice` descending.

</details>

<details>
<summary>Hint 2</summary>

After adding the `quartile` column with `ntile`, do a `.groupBy("franchiseID", "quartile").agg(F.count("*").alias("transaction_count"))` to get the final result.

</details>

## Problem 3

<details>
<summary>Hint 1</summary>

`F.percent_rank()` and `F.cume_dist()` both go over the same window spec: `Window.partitionBy("franchiseID").orderBy("totalPrice")`. The key difference is that `percent_rank` starts at 0 for the first row, while `cume_dist` is always > 0.

</details>

## Problem 4

<details>
<summary>Hint 1</summary>

The window needs `orderBy("dateTime")` so `lead` looks at the next transaction chronologically. Use `F.lead("totalPrice", 1).over(w)` to get `next_price`.

</details>

<details>
<summary>Hint 2</summary>

Compute `price_diff` as `F.col("totalPrice") - F.col("next_price")`. It will naturally be null for the last row per franchise since `next_price` is null there.

</details>

## Problem 5

<details>
<summary>Hint 1</summary>

The key is the frame spec: `Window.partitionBy("franchiseID").orderBy("dateTime").rowsBetween(-2, 0)`. This gives you the current row plus the 2 preceding rows (physical positions, not value-based).

</details>

<details>
<summary>Hint 2</summary>

Apply `F.avg("totalPrice").over(w)` with that window spec. For the first row in a partition the average is just itself; for the second it averages 2 rows, etc.

</details>

## Problem 6

<details>
<summary>Hint 1</summary>

`rangeBetween` uses the ordering column's values, not row positions. Use `Window.partitionBy("pickup_zip").orderBy("tpep_pickup_datetime").rangeBetween(Window.unboundedPreceding, Window.currentRow)` with `F.sum("fare_amount")`.

</details>

## Problem 7

<details>
<summary>Hint 1</summary>

You need the full-partition frame: `rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)`. Without this, `F.first()` and `F.last()` only see up to the current row by default.

</details>

<details>
<summary>Hint 2</summary>

After adding `first_price` and `last_price` columns via the window, use `.select("franchiseID", "first_price", "last_price").dropDuplicates()` to collapse to one row per franchise.

</details>

## Problem 8

<details>
<summary>Hint 1</summary>

Define three separate window specs:
- `w1 = Window.partitionBy("franchiseID").orderBy(F.col("totalPrice").desc())` for `row_number`
- `w2 = Window.partitionBy("franchiseID")` (no ordering) for `sum`
- `w3 = Window.partitionBy("franchiseID").orderBy("dateTime").rowsBetween(-2, 0)` for moving avg

</details>

<details>
<summary>Hint 2</summary>

Chain all three `.withColumn` calls, then `.filter(F.col("rn") <= 5)` and `.select()` to keep only the 6 expected columns.

</details>